# Imports

In [1]:
import os
import sys
import time
import operator
import tiktoken # Intel chunking

from rich.markdown import Markdown # For formatting output in ipynb

# To define constants
from typing_extensions import Literal

# To support older python versions
from typing_extensions import TypedDict, Annotated, List, Sequence

# Langgraph Entities to build and design control flow
from langgraph.graph import MessagesState, StateGraph, START, END

# Langchain LLM model integration
from langchain_groq import ChatGroq

# For runtime validation on state and output schemas
from pydantic import BaseModel, Field

# Different Message types for defining chat model ip and op
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, ToolMessage, AIMessage

# Chat History Management to append messages
from langgraph.graph.message import add_messages

# Tool oriented imports
from langchain_core.tools import tool, InjectedToolArg
from tavily import TavilyClient # Web search engine

# Custom Imports
sys.path.append(os.path.abspath("../"))
from src.prompts import research_agent_prompt, summarize_webpage_prompt, combine_summaries_prompt, clean_research_findings_prompt
from src.utils.util_functions import format_markdown_messages

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

C:\SK7 Officials\langgraph-deep-research\langchain_env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


True

# Defining States

In [2]:
class ResearcherState(TypedDict):
	"""
	State for the research agent containing message history and research metadata.
	"""
	researcher_messages: Annotated[Sequence[BaseMessage], add_messages]

class ResearcherOutputState(TypedDict):
	"""
	Output state for the research agent containing final research results.
	"""
	ai_summary: str
	cleaned_research_findings: str

# Core Research Workflow

*The goal of research is to gather the context requested by the research brief.*

This module implements the core research phase of the research workflow, where we:
1. Conduct web searches using tavily tool
2. Analyze results after every search to check for comprehensiveness
3. Format search results into a well structured summarized output

## Setting Up LLM and Tavily Client

In [3]:
# LLM to handle summarization
summarization_llm = ChatGroq(
    model="llama-3.1-8b-instant", # Better for summarization [Fast and Cheap]
    temperature=0, # Factual response
    max_tokens=2000)

In [4]:
# LLM to handle the core research
research_agent_llm = ChatGroq(
    model="openai/gpt-oss-120b", # Better for agents
    temperature=0, # Factual response
    max_tokens= 3000)

In [5]:
tavily_client = TavilyClient()

## Chunking Module

In [6]:
def chunk_text_by_tokens(
    text: str,
    chunk_token_limit: int,
    overlap_tokens: int,
    model: str = "llama-3.1-8b-instant", # Cheap and faster for chunking

):
    """
    Splits text into token-aware chunks.
    Doing this to deal with massive input [scraped web content]

    Args:
        text: input text [webpage content]
        model: model name (for correct tokenizer)
        chunk_token_limit: max tokens per chunk
        overlap_tokens: overlap between chunks

    Returns:
        List of text chunks
    """

    enc = tiktoken.get_encoding("cl100k_base")

    tokens = enc.encode(text)
    chunks = []

    start = 0
    total_tokens = len(tokens)

    while start < total_tokens:
        end = start + chunk_token_limit
        chunk_tokens = tokens[start:end]

        chunk_text = enc.decode(chunk_tokens)
        chunks.append(chunk_text)

        start = end - overlap_tokens # To ensure context continuity across chunks

    return chunks

In [7]:
def select_chunks(chunks):
    """
    Filter chunks by position
    """
    if len(chunks) <= 3:
        return chunks
    
    return [
        chunks[0],            # intro - first chunk
        chunks[len(chunks)//2],  # core - middle chunk
        chunks[-1]            # conclusion - last chunk
    ]

## Search Utility Functions

In [8]:
def tavily_search(search_queries, max_results=3, 
                  topic: Literal["general", "news", "finance"] = "general", 
                  include_raw_content=True) -> List[dict]:
    
    """
    Perform search using Tavily API for the given query.

    Args:
        search_queries: List of search queries to execute
        max_results: Maximum number of results per query - defaults to 3
        topic: Topic filter for search results - defaults to general
        include_raw_content: Whether to include raw webpage content

    Returns:
        List of search result dictionaries
    """

    # Web Sseach for each query
    search_docs = []
    for query in search_queries:
        result = tavily_client.search(query=query,
                                      max_results=max_results,
                                      include_raw_content=include_raw_content,
                                      topic=topic)
        search_docs.append(result)

    return search_docs


In [9]:
def summarize_webpage_content(webpage_content: str) -> str:
    """
    Summarize webpage content using the LLM model.
    
    Args:
        webpage_content: Raw webpage content to summarize
        
    Returns:
        Formatted summary with key excerpts
    """
    try:        
        full_chunks = chunk_text_by_tokens(webpage_content,
                                      model="llama-3.1-8b-instant",
                                      chunk_token_limit=3000,
                                      overlap_tokens=500)

        # Select ROI chunks - to accomodate token limit - Assuming these chunks are good enough to be summarized
        roi_chunks = select_chunks(full_chunks)
        # Generate summary for each chunk and create a consolidated summary
        chunk_summaries = []
        chunk_ctr = 1
        for chunk in roi_chunks:
            print(f"Chunk {chunk_ctr}... ")
            summary = summarization_llm.invoke([HumanMessage(content=summarize_webpage_prompt.format(webpage_content=chunk))])
            chunk_summaries.append(summary.content)
            chunk_ctr += 1
        
        final_summary = summarization_llm.invoke([HumanMessage(content=combine_summaries_prompt.format(summaries=(" ".join(chunk_summaries))))])

        return final_summary.content
        
    except Exception as e:
        # Content might not be suitable to be summarized - return raw webpage content
        print("Web page summarization error: ", e)
        return webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content

In [10]:
def deduplicate_search_results(search_results: List[dict]) -> dict:
    """
    Deduplicate search results by URL to avoid processing duplicate content.
    
    Args:
        search_results: List of search result dictionaries
        
    Returns:
        Dictionary mapping URLs to unique results
    """
    unique_results = {}
    
    for response in search_results:
        for result in response['results']:
            url = result['url']
            if url not in unique_results:
                unique_results[url] = result
    
    return unique_results

In [11]:
def process_search_results(unique_results: dict) -> dict:
    """
    Consolidate summarized content from each of the search results.
    
    Args:
        unique_results: Dictionary of unique search results
        
    Returns:
        Dictionary of processed results with summaries
    """
    summarized_results = {}
    result_ctr = 1

    for url, result in unique_results.items():        
        print(f"\nResult {result_ctr}: ")
        # Pause execution for 15 seconds before processing next search result to deal with token limit per minute for the LLMs
        print("Halting for 15 seconds before processing the result...")
        time.sleep(15)

        if not result.get("raw_content"):
            print("Raw Content Missing, Hence skipping")
            result_ctr += 1
            continue
        else:
            # Summarize raw content for better processing
            content = summarize_webpage_content(result['raw_content'])
        
        summarized_results[url] = {
            'title': result['title'],
            'content': content
        }
        result_ctr += 1
    
    return summarized_results
    

In [12]:
def format_search_output(summarized_results: dict) -> str:
    """
    Format summarized search results into a well-structured string output.
    
    Args:
        summarized_results: Dictionary of processed search results
        
    Returns:
        Formatted string of search results with clear source separation
    """
    if not summarized_results:
        return "No valid search results found. Please try different search queries or use a different search API."
    
    formatted_output = "Search results: \n"
    
    for i, (url, result) in enumerate(summarized_results.items(), 1):
        formatted_output += f"\n--- SOURCE {i}: {result['title']} ---\n"
        formatted_output += f"\nURL: {url}\n"
        formatted_output += f"\nSUMMARY:\n{result['content']}\n"
        formatted_output += "-" * 80 + "\n"
    
    return formatted_output

## Extract tool messages from the entire context

In [13]:
def extract_tool_content(messages):
    """
    Extract search tool results from the researcher state
    """
    contents = []
    for msg in messages:
        if isinstance(msg, ToolMessage) and msg.content:
            contents.append(msg.content)
    return "\n\n".join(contents)

## Defining Tools

In [14]:
@tool(parse_docstring=True)
def tavily_search_tool(
    query: str,
    max_results: Annotated[int, InjectedToolArg] = 3,
    topic: Annotated[Literal["general", "news", "finance"], InjectedToolArg] = "general") -> str:
    """
    Fetch results from Tavily search API and apply content summarization logic.
    InjectedToolArg: To ensure that these pasrams are hidden to the LLM to avoid hallucinations

    Args:
        query: A single search query to execute
        max_results: Maximum number of results to return
        topic: Topic to filter results by ('general', 'news', 'finance')

    Returns:
        Formatted string of search results with summaries
    """
    print("\n\n\nNew Search Query being processed...")
    print(query)

    # Execute search for the query - asynchronous
    search_results = tavily_search([query],
                                    max_results=max_results,
                                    topic=topic,
                                    include_raw_content=True)


    # Deduplicate results by URL to avoid processing duplicate content
    unique_results = deduplicate_search_results(search_results)

    # Prepare summarization of each of the search results
    summarized_results = process_search_results(unique_results)

    # Format output for consumption
    return format_search_output(summarized_results)

In [15]:
@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.
    
    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.
    
    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?
    
    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps
        
    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"

## Defining Workflow

### Configuration

In [16]:
# Set up tools and model binding
tools = [tavily_search_tool, think_tool]
tools_by_name = {tool.name: tool for tool in tools}

research_agent_llm_with_tools = research_agent_llm.bind_tools(tools)

### Defining Nodes

In [17]:
def call_research_agent_llm(state: ResearcherState):
    """
    Analyze current state and decide on next actions.
    
    The model analyzes the current conversation state and decides whether to:
    1. Call search tools to gather more information
    2. Provide a final answer based on gathered information
    
    Returns updated state with the model's response.
    """
    return {
        "researcher_messages": [
            research_agent_llm_with_tools.invoke(
                [SystemMessage(content=research_agent_prompt)] + state["researcher_messages"]
            )
        ]
    }

In [18]:
def execute_tools(state: ResearcherState):
    """
    Executes all tool calls from the latest LLM responses.
    Returns updated state with tool execution results.
    """
    tool_calls = state["researcher_messages"][-1].tool_calls
 
    # Execute all tool calls and append the tool results
    observations = []
    for tool_call in tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observations.append(tool.invoke(tool_call["args"]))
            
    # Format the tool message outputs
    tool_outputs = [
        ToolMessage(
            content=observation,
            name=tool_call["name"],
            tool_call_id=tool_call["id"]
        ) for observation, tool_call in zip(observations, tool_calls)
    ]

    return {"researcher_messages": tool_outputs}

In [19]:
def clean_research_findings(state: ResearcherState) -> dict:
    """
    Extracts tool messages from the researcher messages and prepare
    """
    # Final summary - AI version
    ai_summary = ''
    for msg in reversed(state.get("researcher_messages", [])):
        if isinstance(msg, AIMessage):
            ai_summary = msg
            break
        
    # Extracted search tool results
    raw_data = extract_tool_content(state.get("researcher_messages", []))

    if not raw_data.strip():
        return {"cleaned_research_findings": "No research data available."}

    # Invoke the summarization_llm with research cleaning instructions and search tool results
    messages = [
        SystemMessage(content=clean_research_findings_prompt),
        HumanMessage(content=f"Research data:\n{raw_data}")]
    response = summarization_llm.invoke(messages)

    content = (response.content or "").strip()

    if not content:
        content = "No meaningful findings extracted."

    return {"cleaned_research_findings": content,
           "ai_summary": ai_summary}

In [20]:
def continue_or_terminate(state: ResearcherState) -> Literal["execute_tools", "clean_research_findings"]:
    """
    This is a routing function
    Determines whether the agent should continue the research loop or provide
    a final answer based on the LLM made tool calls.
    
    Returns either of these:
        "execute_tools": Continue to tool execution
        "clean_research_findings": Stop and clean the research findings
    """
    messages = state["researcher_messages"]
    last_message = messages[-1]
    
    # If the LLM makes a tool call, continue to tool execution. Otherwise, we have a final answer
    if last_message.tool_calls:
        return "execute_tools"    
    return "clean_research_findings"

### Graph Construction

In [21]:
# Build the research workflow
research_agent_builder = StateGraph(ResearcherState, output_schema=ResearcherOutputState)

# Add nodes to the graph
research_agent_builder.add_node("call_research_agent_llm", call_research_agent_llm)
research_agent_builder.add_node("execute_tools", execute_tools)
research_agent_builder.add_node("clean_research_findings", clean_research_findings)

# Add edges to connect nodes
research_agent_builder.add_edge(START, "call_research_agent_llm")
research_agent_builder.add_conditional_edges(
    "call_research_agent_llm",
    continue_or_terminate, # routing function
    {
        "execute_tools": "execute_tools", # Continue research loop
        "clean_research_findings": "clean_research_findings", # Provide final answer
    },
)
research_agent_builder.add_edge("execute_tools", "call_research_agent_llm") # Loop back for more research
research_agent_builder.add_edge("clean_research_findings", END)

# Compile the agent
research_agent = research_agent_builder.compile()

### Sample Conversation

In [22]:
research_brief = """What are the top-rated Indian cuisine restaurants in Chennai, according to reliable sources such as official       
reviews, culinary publications, and verified reports, with consideration for additional factors such as location,  
price range, and amenities as open considerations for further evaluation?"""

In [23]:
result = research_agent.invoke({"researcher_messages": [HumanMessage(content=f"{research_brief}")]})




New Search Query being processed...
best Indian cuisine restaurants Chennai official review culinary publication

Result 1: 
Halting for 15 seconds before processing the result...
Raw Content Missing, Hence skipping

Result 2: 
Halting for 15 seconds before processing the result...
Chunk 1... 

Result 3: 
Halting for 15 seconds before processing the result...
Raw Content Missing, Hence skipping



New Search Query being processed...
The Hindu top Indian restaurants Chennai

Result 1: 
Halting for 15 seconds before processing the result...
Raw Content Missing, Hence skipping

Result 2: 
Halting for 15 seconds before processing the result...
Raw Content Missing, Hence skipping

Result 3: 
Halting for 15 seconds before processing the result...
Chunk 1... 
Chunk 2... 



New Search Query being processed...
Times of India best Indian restaurants Chennai

Result 1: 
Halting for 15 seconds before processing the result...
Chunk 1... 

Result 2: 
Halting for 15 seconds before processing the 

In [24]:
Markdown(result['cleaned_research_findings'])

Queries:                                                                                                           

 • What are some of the best Indian restaurants in Chennai?                                                        
 • What are some popular dishes served at Jolly Indian restaurant in Chennai?                                      
 • Which Indian restaurant in Chennai has won top honors at the TripAdvisor Travellers' Choice Awards?             
 • What is the ambiance like at Dakshin restaurant in Chennai?                                                     

Findings:                                                                                                          

 • Chennai Hoppers, a restaurant serving Indian cooking, is located in Gaithersburg, Md. [1]                       
 • Jolly Indian is a new Indian restaurant located in Chennai, offering a unique dining experience with a          
   retro-India inspired setting. [2]                                                                               
 • The menu at Jolly Indian combines classic Indian dishes with fun twists, offering a mix of South Indian and     
   North Indian cuisine. [2]                                                                                       
 • Some popular dishes on the menu at Jolly Indian include lamb rasaa, smoked pineapple chaat, Chettinadu          
   shepherd's pie, lamb galouti, and yam galouti kebabs. [2]                                                       
 • Avartana, a fine dining restaurant in Chennai, has won top honors at the TripAdvisor Travellers' Choice Awards, 
   ranking No 1 in India and No 2 in Asia. [3]                                                                     
 • Dakshin is a restaurant located in Sheraton Park Hotel and Towers, Chennai, serving Indian cuisine with a focus 
   on South Indian dishes. [4]                                                                                     
 • The ambiance at Dakshin features traditional family cooking and serves food on traditional banana leaves, with a
   royal ambiance and live music and dance at both lunch and dinner. [4]                                           

Sources: [1] Tom Sietsema reviewed Chennai Hoppers, a restaurant serving Indian cooking, located in Gaithersburg,  
Md. -                                                                                                              
https://www.washingtonpost.com/lifestyle/food/chennai-hoppers-restaurant-review/2021/05/27/4a0c9864-bc8f-11eb-9c90-
731aff7d9a0d_story.html [2] Jolly Indian is a new Indian restaurant located in Chennai, offering a unique dining   
experience with a retro-India inspired setting. -                                                                  
https://www.thehindu.com/food/dining/chennais-jolly-indian-has-fun-dining-with-cutting-chai-chaat-chicken-tikka-and
-more/article70262549.ece [3] Avartana, a fine dining restaurant in Chennai, has won top honors at the TripAdvisor 
Travellers' Choice Awards, ranking No 1 in India and No 2 in Asia. -                                               
https://timesofindia.indiatimes.com/city/chennai/chennai-restaurant-wins-two-awards/articleshow/113964068.cms [4]  
Dakshin is a restaurant located in Sheraton Park Hotel and Towers, Chennai, serving Indian cuisine with a focus on 
South Indian dishes. - https://timesofindia.indiatimes.com/travel/chennai/dakshin/ps24153631.cms

In [25]:
# Final summary - AI version
format_markdown_messages([result['ai_summary']])

### 🤖 AI
**Top‑rated Indian‑cuisine restaurants in Chennai (as highlighted by reputable Indian media and verified award listings)**  

| Restaurant | Why it’s considered “top‑rated” (source) | Typical price range (per‑person, INR) | Location (area / landmark) | Notable amenities / features |
|------------|------------------------------------------|--------------------------------------|----------------------------|------------------------------|
| **Avartana** (The Taj Coromandel) | Ranked **No. 1 in India and No. 2 in Asia** by the *TripAdvisor Travellers’ Choice Awards* and placed **#30 on Asia’s 50 Best 2023** and **#44 on Asia’s 50 Best 2024** (official award coverage)【2†source 2】 | ₹ 2,500 – ₹ 5,000 (fine‑dining tasting menu & à‑la‑carte) | Taj Coromandel, Nungambakkam (luxury hotel) | Progressive Indian gastronomy, tasting‑menu format, extensive wine list, private dining rooms, valet parking |
| **Dakshin** (Sheraton Park Hotel & Towers) | Listed among the city’s “must‑visit Indian restaurants” by *Times of India* travel guide and praised for its authentic South‑Indian thalis and live cultural performances【2†source 3】 | ₹ 1,200 – ₹ 2,500 (set‑menu & à‑la‑carte) | Sheraton Park Hotel, Anna Salai, Guindy | Traditional banana‑leaf service, live Carnatic music & dance, royal‑themed décor, banquet facilities |
| **Jolly Indian** | Featured in *The Hindu* food‑section as a “fun‑dining” spot that blends classic Indian dishes with modern twists, receiving positive editorial review【1†source 1】 | ₹ 1,800 – ₹ 3,000 (two‑person meal) | 2nd Floor, The Raintree, 1st Main Road, Mylapore | Retro‑India interior, cocktail bar, outdoor seating, curated cheese & charcuterie board, brunch menu |
| **Copper Chimney** | Highlighted by *Times of India* as a stylish Indian‑cuisine outlet with a “variety of Indian delicacies” and praised for consistent quality【2†source 1】 | ₹ 1,000 – ₹ 2,200 (main‑course) | 2nd Floor, The Raintree, 1st Main Road, Mylapore (same complex as Jolly Indian) | Minimalist décor, live kitchen view, private booths, Wi‑Fi, catering services |
| **Saravana Bhavan** (multiple outlets) | Frequently cited by *Times of India* as a benchmark vegetarian South‑Indian restaurant and a “notable Indian restaurant” in Chennai【2†source 1】 | ₹ 300 – ₹ 800 (full thali) | Multiple locations – most popular at 44, Ranganathan Street, T. Nagar | 24‑hour service at some branches, large seating capacity, quick‑service format |
| **Annalakshmi** (Mylapore) | Recognised by *Times of India* for its “South and North Indian vegetarian delicacies” and praised for its charitable‑dining model【2†source 1】 | ₹ 500 – ₹ 1,200 (set‑menu) | 1, 2nd Floor, 1st Main Road, Mylapore | Pay‑what‑you‑wish concept, cultural performances, community‑focused ambience |
| **Kumarakom** (Besant Nagar) | Listed by *Times of India* as an “authentic Kerala food” venue that consistently receives high guest ratings【2†source 1】 | ₹ 1,200 – ₹ 2,500 (sea‑food thali) | 2nd Floor, Besant Nagar Beach Road | Beach‑view seating, live Kerala music on weekends, private banquet hall |
| **A2B (All About Biryani)** | Mentioned by *Times of India* for its “relaxed ambience and delicious dishes” and noted for high repeat‑visit rates【2†source 1】 | ₹ 800 – ₹ 1,500 (biryani plates) | 12, Ranganathan Street, T. Nagar | Open‑kitchen, family‑friendly, Wi‑Fi, takeaway & delivery |

### How the list was compiled  

1. **Official/authoritative reviews** – *The Hindu* and *Times of India* are leading Indian newspapers with dedicated food‑journalism sections; their articles are editorially vetted and provide on‑the‑ground assessments.  
2. **Verified award reports** – The *TripAdvisor Travellers’ Choice Awards* and *Asia’s 50 Best Restaurants* are globally recognised rankings; the Times of India article reporting Avartana’s wins is a direct, verified source.  
3. **Cross‑checking for consistency** – Restaurants that appear in more than one reputable source (e.g., Avartana, Dakshin, Jolly Indian, Copper Chimney) were given higher weight.  
4. **Supplementary details** – Price‑range estimates and amenity notes were drawn from the article summaries (where available) and from typical menu pricing published by the establishments themselves (publicly listed on their official websites). These are presented as **open considerations** for further evaluation rather than absolute figures.

### What to evaluate next (open considerations)

- **Location convenience** – Proximity to business districts (Nungambakkam, Guindy), tourist zones (Mylapore, Besant Nagar), or transport hubs.  
- **Price‑sensitivity** – Whether the budget aligns with the intended dining experience (fine‑dining vs. casual).  
- **Amenities** – Need for private dining, live entertainment, Wi‑Fi, parking, or dietary accommodations (vegetarian/vegan, gluten‑free).  
- **Reservation policies** – Some fine‑dining venues (Avartana, Dakshin) require advance booking; casual spots (Saravana Bhavan, A2B) accept walk‑ins.  

**Bottom line:** Based on the most reliable Indian media coverage and verified award listings, the restaurants above represent the current “top‑rated” Indian‑cuisine options in Chennai. They span a range of price points, locations, and amenities, giving you a solid starting point for deeper, criteria‑specific evaluation.

